# Extracting Frames from Deep Sea Video

This notebook walks through how to use `soi-frame-extractor` to pull still images from video files, with sensor data (position, depth, temperature, salinity, etc.) embedded directly into each image's metadata.

**What you need:**
- One or more video files (`.mp4` or `.mov`) in a folder
- Optionally: a CSV file from your vehicle's data system with timestamps and sensor readings
- An extraction spec: a small text file describing how often to extract frames and any conditions to apply

**What you get:**
- JPEG images in an output folder, named by timestamp
- Sensor metadata embedded into each image (GPS coordinates, depth, etc.)
- An `ifdo.json` metadata file describing the full dataset
- A formatted `biigle_metadata.csv` ready to upload to BIIGLE with your frames.

## Installation

Run this once in your terminal before opening this notebook:

```
uv pip install git+https://github.com/schmidtocean/soi-frame-extractor
```

If `uv` is not installed, see https://docs.astral.sh/uv/getting-started/installation/ or install using pip:

```
git clone https://github.com/schmidtocean/soi-frame-extractor
cd soi-frame-extractor
pip install -e .
```

## The Extraction Spec

The extraction spec is a plain text file (YAML format) that tells the tool how to extract frames. Save it as `spec.yaml` in the same folder as this notebook, or update the path in the cell above.

Here is an example minimal spec to extract one frame every 10 seconds:

```yaml
rules:
  - interval_s: 10.0
```

A fuller example with sensor data and project metadata might look like:

```yaml
rules:
  - interval_s: 10.0

mappings:
  timestamp:  Timestamp        # the column in your CSV that holds the time
  latitude:   Latitude_ddeg    # adjust these to match your CSV column names
  longitude:  Longitude_ddeg
  depth:      Depth_m

metadata:
  cruise_id: FKt250101
  dive_id:   S0042
  vehicle:   SuBastian
```

The `mappings` block is required when you want to include sensor data in a CSV file.

The `metadata` block is optional. Any values you include will be embedded into the extracted frames.

A complete example of all available spec options is available in the repository as `extraction_spec.yaml`.
Note that the CSV mapping section writes specific fields (latitude, longitude, depth, etc.) to the correct
fields in EXIF, iFDO, and BIIGLE CSV automatically. See `extraction_spec.yaml` for more details and
consult the Advanced Extraction notebook for more detailed examples.

## Example 1: Extract frames at a fixed interval

In this example we will extract frames once every 10 seconds from a video file of your choice.

Create a spec YAML with a simple 10-second interval rule as:

```yaml
rules:
  - interval_s: 10.0
```

Edit the code block below to set directories and spec location:

In [ ]:
from pathlib import Path

# Point these at your actual files and folders.
# Path("video/") means a folder called "video" in the same directory as this notebook.
# You can also use an absolute path, e.g. Path("/home/thomas/data/dive42/video/")

video_dir  = Path("video/")      # folder containing your .mp4 or .mov files
spec_file  = Path("spec.yaml")   # your extraction spec (see below)
output_dir = Path("frames/")     # where extracted frames will be written

# Create the output folder if it doesn't exist yet
output_dir.mkdir(exist_ok=True)

This code block will perform the extraction by calling `extract` with the spec file,
source, and destination directories:

In [ ]:
from soi_frame_extractor import extract

extract(
    spec_path=spec_file,
    video_source=video_dir,
    output_dir=output_dir,
)

frames = sorted(output_dir.glob("*.jpg"))
print(f"Done. {len(frames)} frames written to {output_dir}/")

## What's in the output folder?

After running you will find:

| File | What it is |
|---|---|
| `*.jpg` | Extracted frames, one per planned timestamp |
| `ifdo.json` | Dataset manifest — one entry per frame with all metadata |
| `biigle_metadata.csv` | Metadata CSV ready to upload directly to BIIGLE |

## Example 2: With sensor data

If you have a CSV from your vehicle's data system, pass it to `extract()` with `csv_path`. The tool will interpolate sensor values linearly at each frame's exact timestamp and embed them in the image.

Make sure your `spec.yaml` includes a `mappings` block that matches your CSV column names. As an example, if your CSV file was

```csv
Timestamp, Latitude_ddeg, Longitude_ddeg, Depth_m, heading
2025-10-15T10:10:00.278262Z,-44.28954398,-59.9191321,375.2,88.1323
2025-10-15T10:10:21.278142Z,-44.28954394,-59.9193453,376.8,88.1378
2025-10-15T10:10:42.282541Z,-44.28954394,-59.91913218,379.3,88.1378
```

The corresponding spec file will require a mappings block to handle the headings in the csv. Some mapping keys are keywords and map to specific fields in the EXIF and BIIGLE metadata. Other columns not in the key mappings list will default to the XMP metadata output. Timestamp is always required for any sensor data:

```yaml
rules:
  - interval_s: 10.0

mappings:
  timestamp:  Timestamp        # the column in your CSV that holds the time
  latitude:   Latitude_ddeg    # adjust these to match your CSV column names
  longitude:  Longitude_ddeg
  depth:      Depth_m
  heading:    heading

```

Point the `csv_file` variable to your sensor CSV path and run the extraction.

In [ ]:
csv_file = Path("data/sensors.csv")   # update this to your sensor CSV

extract(
    spec_path=spec_file,
    video_source=video_dir,
    output_dir=output_dir,
    csv_path=csv_file,
)

frames = sorted(output_dir.glob("*.jpg"))
print(f"Done. {len(frames)} frames written to {output_dir}/")

Sensor values (depth, position, etc.) are embedded inside each JPEG, they travel with the image into any downstream tool that reads EXIF or XMP. They are also written to the biggle_metadata.csv file in the frames directory. To confirm, you can examine the BIIGLE file or open an image with a tool that reads EXIF/XMP data.